# State Feedback and Output Feedback (Interactive)

This notebook builds a 2-state LTI model and explores output feedback using a root locus and impulse response.

Dimensions used in this notebook:

- $A \in \mathbb{R}^{2\times 2}$
- $B \in \mathbb{R}^{2\times 1}$
- $C \in \mathbb{R}^{1\times 2}$

## 1. Regular LTI System Equations

We consider the continuous-time LTI system

$$
\dot{x}(t) = A x(t) + B u(t), \qquad y(t) = C x(t),
$$

with

$$
x(t) \in \mathbb{R}^2,\quad u(t) \in \mathbb{R},\quad y(t) \in \mathbb{R}.
$$

You can set the matrices $A$, $B$, and $C$ below using text boxes.

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from IPython.display import display, Markdown

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.grid'] = True

In [8]:
def parse_matrix(text, shape):
    vals = [v.strip() for v in text.split(',') if v.strip() != '']
    expected = shape[0] * shape[1]
    if len(vals) != expected:
        raise ValueError(f'Expected {expected} values, got {len(vals)}')
    arr = np.array([float(v) for v in vals], dtype=float).reshape(shape)
    return arr

A_text = widgets.Text(
    value='0, 1, -2, -3',
    description='A:',
    layout=widgets.Layout(width='450px')
)
B_text = widgets.Text(
    value='0, 1',
    description='B:',
    layout=widgets.Layout(width='450px')
)
C_text = widgets.Text(
    value='1, 0',
    description='C:',
    layout=widgets.Layout(width='450px')
)

out_model = widgets.Output()


def update_model(_=None):
    out_model.clear_output()
    with out_model:
        try:
            A = parse_matrix(A_text.value, (2, 2))
            B = parse_matrix(B_text.value, (2, 1))
            C = parse_matrix(C_text.value, (1, 2))

            print('Parsed matrices:')
            print('A =')
            print(A)
            print('B =')
            print(B)
            print('C =')
            print(C)

            eigA = np.linalg.eigvals(A)
            print(f'Open-loop poles (eig(A)): {eigA}')

            # Store globally for the output-feedback section.
            globals()['A_current'] = A
            globals()['B_current'] = B
            globals()['C_current'] = C

        except Exception as e:
            print(f'Input error: {e}')

A_text.observe(update_model, names='value')
B_text.observe(update_model, names='value')
C_text.observe(update_model, names='value')

display(widgets.VBox([
    widgets.HTML('<b>Enter matrices as comma-separated values, row-major order</b>'),
    widgets.HTML('A (2x2): a11, a12, a21, a22'),
    A_text,
    widgets.HTML('B (2x1): b1, b2'),
    B_text,
    widgets.HTML('C (1x2): c1, c2'),
    C_text,
    out_model
]))

update_model()

## 2. Output Feedback: Root Locus and Impulse Response

Use static output feedback

$$
u(t) = -K y(t) + r(t), \qquad y(t)=Cx(t),
$$

which yields the closed-loop matrix

$$
A_{cl}(K) = A - BKC.
$$

The plot below shows:

- Root-locus style map of closed-loop poles as $K$ varies over a range.
- Current closed-loop poles for the selected slider value of $K$.
- Impulse response of the closed-loop transfer from $r$ to $y$.

In [9]:
K_slider = widgets.FloatSlider(
    value=0.0,
    min=-20.0,
    max=20.0,
    step=0.1,
    description='K:',
    continuous_update=False,
    readout_format='.2f'
)

k_scan = np.linspace(-20, 20, 500)


def plot_output_feedback(K):
    try:
        A = globals().get('A_current', np.array([[0.0, 1.0], [-2.0, -3.0]]))
        B = globals().get('B_current', np.array([[0.0], [1.0]]))
        C = globals().get('C_current', np.array([[1.0, 0.0]]))

        if A.shape != (2, 2) or B.shape != (2, 1) or C.shape != (1, 2):
            raise ValueError('Matrix dimensions must be A(2x2), B(2x1), C(1x2).')

        poles_all = []
        for kk in k_scan:
            Acl_scan = A - B @ (kk * C)
            poles_all.extend(np.linalg.eigvals(Acl_scan))
        poles_all = np.array(poles_all)

        Acl = A - B @ (K * C)
        poles_now = np.linalg.eigvals(Acl)

        B_r = B
        D = np.array([[0.0]])
        sys_cl = signal.StateSpace(Acl, B_r, C, D)
        t = np.linspace(0, 15, 800)
        tout, y = signal.impulse(sys_cl, T=t)

        fig, ax = plt.subplots(1, 2, figsize=(13, 5))

        ax[0].scatter(poles_all.real, poles_all.imag, s=7, alpha=0.22, label='Poles for K in scan')
        ax[0].scatter(poles_now.real, poles_now.imag, s=100, marker='x', label=f'Poles at K={K:.2f}')
        ax[0].axhline(0.0, linewidth=1)
        ax[0].axvline(0.0, linewidth=1)
        ax[0].set_title('Root-Locus Style Pole Map')
        ax[0].set_xlabel('Real part')
        ax[0].set_ylabel('Imaginary part')
        ax[0].legend(loc='best')

        ax[1].plot(tout, np.squeeze(y), linewidth=2)
        ax[1].set_title('Impulse Response (r -> y)')
        ax[1].set_xlabel('Time [s]')
        ax[1].set_ylabel('y(t)')

        plt.tight_layout()
        plt.show()

        print(f'Closed-loop poles at K={K:.2f}: {poles_now}')

    except Exception as e:
        print(f'Plot error: {e}')

widgets.interact(plot_output_feedback, K=K_slider);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='K:', max=20.0, min=-20.0), …

## 3. State Feedback Power: Two Gains + Click-to-Place Poles

Now use full state feedback (no feedforward term yet):

$$
u(t) = -Kx(t), \qquad K = \begin{bmatrix}k_1 & k_2\end{bmatrix}.
$$

Then

$$
A_{cl}(k_1,k_2) = A - BK.
$$

In the interactive plot below:

- Move sliders for $k_1$ and $k_2$.
- Click on the pole map to choose a target pole location.
- The notebook computes $(k_1,k_2)$ that place the closed-loop poles at that clicked complex-conjugate pair (when possible).
- The impulse response (from impulse disturbance at the input channel to output $y$) updates next to the pole map.

In [10]:
# Enable an interactive matplotlib backend so click events on the pole map are captured.
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
    _sf_backend_ok = True
except Exception:
    _sf_backend_ok = False

k1_slider = widgets.FloatSlider(value=0.0, min=-100.0, max=100.0, step=0.1, description='k1:', continuous_update=False)
k2_slider = widgets.FloatSlider(value=0.0, min=-100.0, max=100.0, step=0.1, description='k2:', continuous_update=False)

# Text boxes allow entering very large gains directly.
k1_text = widgets.FloatText(value=0.0, description='k1 exact:')
k2_text = widgets.FloatText(value=0.0, description='k2 exact:')

info_html = widgets.HTML('Click in the pole map to auto-place poles.')
out_sf = widgets.Output()

_sf_updating = {'busy': False}


def _poly_coeff_affine_map(A, B):
    def coeffs(k1, k2):
        K = np.array([[k1, k2]], dtype=float)
        Acl = A - B @ K
        p = np.poly(Acl)
        return np.array([p[1], p[2]], dtype=float)

    c0 = coeffs(0.0, 0.0)
    c1 = coeffs(1.0, 0.0) - c0
    c2 = coeffs(0.0, 1.0) - c0
    M = np.column_stack([c1, c2])
    return c0, M


def _ensure_slider_range(slider, value):
    # Auto-expand bounds so large computed gains remain reachable.
    if value < slider.min:
        slider.min = min(value * 1.2, value - 1.0)
    if value > slider.max:
        slider.max = max(value * 1.2, value + 1.0)


def _ordered_branches(eigs_seq):
    # Track branch continuity by nearest-neighbor assignment between successive eigenvalue pairs.
    b1 = [eigs_seq[0][0]]
    b2 = [eigs_seq[0][1]]
    for i in range(1, len(eigs_seq)):
        c0, c1 = eigs_seq[i][0], eigs_seq[i][1]
        p0, p1 = b1[-1], b2[-1]
        d_keep = abs(c0 - p0) + abs(c1 - p1)
        d_swap = abs(c1 - p0) + abs(c0 - p1)
        if d_keep <= d_swap:
            b1.append(c0)
            b2.append(c1)
        else:
            b1.append(c1)
            b2.append(c0)
    return np.array(b1), np.array(b2)


def _continuous_pole_curves(A, B, k1, k2):
    npts = 700

    k1_vals = np.linspace(k1_slider.min, k1_slider.max, npts)
    eigs_k1 = []
    for g1 in k1_vals:
        K = np.array([[g1, k2]], dtype=float)
        eigs_k1.append(np.linalg.eigvals(A - B @ K))
    b1_k1, b2_k1 = _ordered_branches(eigs_k1)

    k2_vals = np.linspace(k2_slider.min, k2_slider.max, npts)
    eigs_k2 = []
    for g2 in k2_vals:
        K = np.array([[k1, g2]], dtype=float)
        eigs_k2.append(np.linalg.eigvals(A - B @ K))
    b1_k2, b2_k2 = _ordered_branches(eigs_k2)

    return (b1_k1, b2_k1), (b1_k2, b2_k2)


def _set_k_values(k1, k2):
    _sf_updating['busy'] = True

    _ensure_slider_range(k1_slider, k1)
    _ensure_slider_range(k2_slider, k2)

    k1_slider.value = float(k1)
    k2_slider.value = float(k2)
    k1_text.value = float(k1)
    k2_text.value = float(k2)

    _sf_updating['busy'] = False


def _draw_state_feedback_view(k1, k2):
    A = globals().get('A_current', np.array([[0.0, 1.0], [-2.0, -3.0]]))
    B = globals().get('B_current', np.array([[0.0], [1.0]]))
    C = globals().get('C_current', np.array([[1.0, 0.0]]))

    if A.shape != (2, 2) or B.shape != (2, 1) or C.shape != (1, 2):
        raise ValueError('Matrix dimensions must be A(2x2), B(2x1), C(1x2).')

    (b1_k1, b2_k1), (b1_k2, b2_k2) = _continuous_pole_curves(A, B, k1, k2)

    K_now = np.array([[k1, k2]], dtype=float)
    Acl = A - B @ K_now
    poles_now = np.linalg.eigvals(Acl)

    sys_cl = signal.StateSpace(Acl, B, C, np.array([[0.0]]))
    t = np.linspace(0, 15, 800)
    tout, y = signal.impulse(sys_cl, T=t)

    fig, ax = plt.subplots(1, 2, figsize=(13, 5))

    ax[0].plot(b1_k1.real, b1_k1.imag, linewidth=1.5, alpha=0.85, label='Sweep k1 (branch 1)')
    ax[0].plot(b2_k1.real, b2_k1.imag, linewidth=1.5, alpha=0.85, label='Sweep k1 (branch 2)')
    ax[0].plot(b1_k2.real, b1_k2.imag, linewidth=1.5, alpha=0.85, linestyle='--', label='Sweep k2 (branch 1)')
    ax[0].plot(b2_k2.real, b2_k2.imag, linewidth=1.5, alpha=0.85, linestyle='--', label='Sweep k2 (branch 2)')
    ax[0].scatter(poles_now.real, poles_now.imag, s=120, marker='x', color='black', label='Current poles')

    ax[0].axhline(0.0, linewidth=1)
    ax[0].axvline(0.0, linewidth=1)
    ax[0].set_title('Continuous Pole Map (click to place)')
    ax[0].set_xlabel('Real part')
    ax[0].set_ylabel('Imaginary part')
    ax[0].legend(loc='best', fontsize=8)

    ax[1].plot(tout, np.squeeze(y), linewidth=2)
    ax[1].set_title('Impulse Response (input -> y)')
    ax[1].set_xlabel('Time [s]')
    ax[1].set_ylabel('y(t)')

    plt.tight_layout()

    def on_click(event):
        if event.inaxes != ax[0] or event.xdata is None or event.ydata is None:
            return

        sigma = float(event.xdata)
        omega = float(abs(event.ydata))

        a1_des = -2.0 * sigma
        a0_des = sigma**2 + omega**2

        try:
            c0, M = _poly_coeff_affine_map(A, B)
            rhs = np.array([a1_des, a0_des]) - c0
            if np.linalg.matrix_rank(M) < 2:
                info_html.value = '<b>Placement failed:</b> this (A,B) pair cannot place arbitrary poles.'
                return

            k_sol = np.linalg.solve(M, rhs)
            _set_k_values(float(k_sol[0]), float(k_sol[1]))
            info_html.value = (
                f'Clicked target: sigma={sigma:.3f}, omega={omega:.3f}. '
                f'Updated gains: k1={k_sol[0]:.3f}, k2={k_sol[1]:.3f}.'
            )
            _update_sf()
        except Exception as e:
            info_html.value = f'<b>Placement failed:</b> {e}'

    fig.canvas.mpl_connect('button_press_event', on_click)
    plt.show()

    print(f'Current K = [k1, k2] = [{k1:.3f}, {k2:.3f}]')
    print(f'Closed-loop poles: {poles_now}')


def _update_sf(_=None):
    if _sf_updating['busy']:
        return
    out_sf.clear_output(wait=True)
    with out_sf:
        try:
            _draw_state_feedback_view(k1_slider.value, k2_slider.value)
        except Exception as e:
            print(f'State-feedback plot error: {e}')


def _on_slider_change(_=None):
    if _sf_updating['busy']:
        return
    _sf_updating['busy'] = True
    k1_text.value = float(k1_slider.value)
    k2_text.value = float(k2_slider.value)
    _sf_updating['busy'] = False
    _update_sf()


def _on_text_change(_=None):
    if _sf_updating['busy']:
        return
    _set_k_values(k1_text.value, k2_text.value)
    _update_sf()


k1_slider.observe(_on_slider_change, names='value')
k2_slider.observe(_on_slider_change, names='value')
k1_text.observe(_on_text_change, names='value')
k2_text.observe(_on_text_change, names='value')

backend_note = 'Matplotlib widget backend active: click-to-place enabled.' if _sf_backend_ok else (
    'Interactive backend not enabled. Install/enable ipympl and rerun this cell for click-to-place.'
)

info_html.value = backend_note

display(widgets.VBox([
    widgets.HBox([k1_slider, k2_slider]),
    widgets.HBox([k1_text, k2_text]),
    info_html,
    out_sf
]))

_update_sf()

## 4. Feedforward for Reference Tracking

Now include feedforward in the control law:

$$
u = -Kx + \bar{N}r.
$$

For a fixed (slow but stable) state-feedback gain $K$, compare:

- No feedforward: $\bar{N}=0$
- Manual feedforward: choose $\bar{N}$ with a slider
- Optimal feedforward:

$$
\bar{N}_{\mathrm{opt}} = -\frac{1}{C\,(A-BK)^{-1}B}
$$

(when the scalar denominator is nonzero).

In [11]:
N_slider = widgets.FloatSlider(value=0.0, min=-0.5, max=0.5, step=0.05, description='Nbar:', continuous_update=False)
use_opt_checkbox = widgets.Checkbox(value=False, description='Use optimal Nbar')
out_ff = widgets.Output()


def _compute_slow_stable_k(A, B):
    desired_poles = np.array([-0.3, -0.5], dtype=float)
    ctrb = np.hstack([B, A @ B])
    if np.linalg.matrix_rank(ctrb) < 2:
        raise ValueError('System is not controllable, cannot place slow stable poles.')
    K = signal.place_poles(A, B, desired_poles).gain_matrix
    return K, desired_poles


def _compute_nbar_opt(Acl, B, C):
    den = float((C @ np.linalg.solve(Acl, B))[0, 0])
    if abs(den) < 1e-10:
        raise ValueError('C(A-BK)^(-1)B is near zero, Nbar_opt is not defined.')
    return -1.0 / den


def _plot_feedforward_section(_=None):
    out_ff.clear_output(wait=True)
    with out_ff:
        try:
            A = globals().get('A_current', np.array([[0.0, 1.0], [-2.0, -3.0]]))
            B = globals().get('B_current', np.array([[0.0], [1.0]]))
            C = globals().get('C_current', np.array([[1.0, 0.0]]))

            if A.shape != (2, 2) or B.shape != (2, 1) or C.shape != (1, 2):
                raise ValueError('Matrix dimensions must be A(2x2), B(2x1), C(1x2).')

            K_slow, desired = _compute_slow_stable_k(A, B)
            Acl = A - B @ K_slow
            nbar_opt = _compute_nbar_opt(Acl, B, C)

            if use_opt_checkbox.value:
                nbar_used = nbar_opt
                N_slider.disabled = True
            else:
                nbar_used = float(N_slider.value)
                N_slider.disabled = False

            t = np.linspace(0, 40, 1200)

            sys_no_ff = signal.StateSpace(Acl, B * 0.0, C, np.array([[0.0]]))
            t1, y_no_ff = signal.step(sys_no_ff, T=t)

            sys_with_ff = signal.StateSpace(Acl, B * nbar_used, C, np.array([[0.0]]))
            t2, y_with_ff = signal.step(sys_with_ff, T=t)

            fig, ax = plt.subplots(1, 2, figsize=(13, 5))

            poles_cl = np.linalg.eigvals(Acl)
            ax[0].scatter(poles_cl.real, poles_cl.imag, s=120, marker='x', color='black')
            ax[0].axhline(0.0, linewidth=1)
            ax[0].axvline(0.0, linewidth=1)
            ax[0].set_title('Closed-Loop Poles (fixed slow K)')
            ax[0].set_xlabel('Real part')
            ax[0].set_ylabel('Imaginary part')

            ax[1].plot(t1, np.squeeze(y_no_ff), linewidth=2, label='No feedforward (Nbar = 0)')
            ax[1].plot(t2, np.squeeze(y_with_ff), linewidth=2, label=f'With feedforward (Nbar = {nbar_used:.3f})')
            ax[1].plot(t, np.ones_like(t), linestyle='--', linewidth=1.5, label='Reference r = 1')
            ax[1].set_title('Step Response y(t) to Reference')
            ax[1].set_xlabel('Time [s]')
            ax[1].set_ylabel('y(t)')
            ax[1].legend(loc='best')

            plt.tight_layout()
            plt.show()

            print(f'Desired slow poles used for K: {desired}')
            print(f'Computed K (row vector): {K_slow}')
            print(f'Closed-loop poles with K: {poles_cl}')
            print(f'Optimal Nbar: {nbar_opt:.6f}')
            if use_opt_checkbox.value:
                print('Using Nbar = Nbar_opt (checkbox enabled).')
            else:
                print(f'Using manual Nbar from slider: {nbar_used:.6f}')

        except Exception as e:
            N_slider.disabled = False
            print(f'Feedforward section error: {e}')


N_slider.observe(_plot_feedforward_section, names='value')
use_opt_checkbox.observe(_plot_feedforward_section, names='value')

display(widgets.VBox([
    widgets.HBox([N_slider, use_opt_checkbox]),
    out_ff
]))

_plot_feedforward_section()

## 5. LQR: Stabilization and Reference Tracking

Tune the LQR weights and analyze both behaviors in one place:

$$
K_{\mathrm{LQR}} = R^{-1} B^T P, \quad
A^T P + P A - P B R^{-1} B^T P + Q = 0.
$$

Control laws used in this section:

- Stabilization: $u = -K_{\mathrm{LQR}}x$
- Reference tracking: $u = -K_{\mathrm{LQR}}x + \bar{N}r$

with automatic feedforward

$$
\bar{N} = -\frac{1}{C\,(A-BK_{\mathrm{LQR}})^{-1}B}.
$$

The plots are shown together in one layout for direct comparison.

In [12]:
from scipy.linalg import solve_continuous_are

Q_text_lqr = widgets.Text(value='10, 0, 0, 1', description='Q:', layout=widgets.Layout(width='420px'))
R_text_lqr = widgets.Text(value='1', description='R:', layout=widgets.Layout(width='220px'))
out_lqr = widgets.Output()


def _parse_R_scalar(text):
    vals = [v.strip() for v in text.split(',') if v.strip() != '']
    if len(vals) != 1:
        raise ValueError('R must be one scalar value for this SISO system.')
    rval = float(vals[0])
    if rval <= 0:
        raise ValueError('R must be > 0.')
    return np.array([[rval]], dtype=float)


def _compute_lqr_gain(A, B, Q, R):
    P = solve_continuous_are(A, B, Q, R)
    K = np.linalg.solve(R, B.T @ P)
    return K, P


def _draw_lqr_combined(_=None):
    out_lqr.clear_output(wait=True)
    with out_lqr:
        try:
            A = globals().get('A_current', np.array([[0.0, 1.0], [-2.0, -3.0]]))
            B = globals().get('B_current', np.array([[0.0], [1.0]]))
            C = globals().get('C_current', np.array([[1.0, 0.0]]))

            Q = parse_matrix(Q_text_lqr.value, (2, 2))
            R = _parse_R_scalar(R_text_lqr.value)

            if not np.allclose(Q, Q.T):
                raise ValueError('Q must be symmetric.')

            K_lqr, _ = _compute_lqr_gain(A, B, Q, R)
            Acl = A - B @ K_lqr
            poles = np.linalg.eigvals(Acl)

            den = float((C @ np.linalg.solve(Acl, B))[0, 0])
            if abs(den) < 1e-10:
                raise ValueError('C(A-BK)^(-1)B is near zero, automatic Nbar is undefined.')
            nbar = -1.0 / den

            t = np.linspace(0, 20, 900)

            # Stabilization: nonzero initial condition, zero external input.
            x0 = np.array([1.0, 0.0])
            sys_stab = signal.StateSpace(Acl, np.zeros((2, 1)), C, np.array([[0.0]]))
            u_zero = np.zeros_like(t)
            t_stab, y_stab, x_stab = signal.lsim(sys_stab, U=u_zero, T=t, X0=x0)
            u_stab = -np.squeeze(K_lqr @ x_stab.T)

            # Tracking: reference step with automatic Nbar.
            sys_track = signal.StateSpace(Acl, B * nbar, C, np.array([[0.0]]))
            r_step = np.ones_like(t)
            t_track, y_track, x_track = signal.lsim(sys_track, U=r_step, T=t, X0=np.zeros(2))
            u_track = -np.squeeze(K_lqr @ x_track.T) + nbar * r_step

            fig, ax = plt.subplots(2, 2, figsize=(13, 9))

            ax[0, 0].scatter(poles.real, poles.imag, s=120, marker='x', color='black')
            ax[0, 0].axhline(0.0, linewidth=1)
            ax[0, 0].axvline(0.0, linewidth=1)
            ax[0, 0].set_title('LQR Closed-Loop Poles')
            ax[0, 0].set_xlabel('Real part')
            ax[0, 0].set_ylabel('Imaginary part')

            ax[0, 1].plot(t_stab, np.squeeze(y_stab), linewidth=2, label='y(t), x(0)=[1,0]')
            ax[0, 1].set_title('Stabilization (u = -Kx)')
            ax[0, 1].set_xlabel('Time [s]')
            ax[0, 1].set_ylabel('y(t)')
            ax[0, 1].legend(loc='best')

            ax[1, 0].plot(t_track, np.squeeze(y_track), linewidth=2, label='y(t) with automatic Nbar')
            ax[1, 0].plot(t, np.ones_like(t), linestyle='--', linewidth=1.5, label='Reference r = 1')
            ax[1, 0].set_title('Reference Tracking (u = -Kx + Nbar r)')
            ax[1, 0].set_xlabel('Time [s]')
            ax[1, 0].set_ylabel('y(t)')
            ax[1, 0].legend(loc='best')

            ax[1, 1].plot(t_stab, np.abs(np.squeeze(u_stab)), linewidth=2, label='|u(t)| stabilization')
            ax[1, 1].plot(t_track, np.abs(np.squeeze(u_track)), linewidth=2, label='|u(t)| tracking')
            ax[1, 1].set_title('Input Magnitude Comparison')
            ax[1, 1].set_xlabel('Time [s]')
            ax[1, 1].set_ylabel('|u(t)|')
            ax[1, 1].legend(loc='best')

            plt.tight_layout()
            plt.show()

            print(f'Q =\n{Q}')
            print(f'R =\n{R}')
            print(f'K_lqr = {K_lqr}')
            print(f'Nbar (auto) = {nbar:.6f}')
            print(f'Poles = {poles}')
            print(f'max|u| stabilization = {float(np.max(np.abs(u_stab))):.6f}')
            print(f'max|u| tracking = {float(np.max(np.abs(u_track))):.6f}')
            print(f'y_track(t_end) ~ {float(np.squeeze(y_track)[-1]):.6f}')

            globals()['K_lqr_current'] = K_lqr
            globals()['Acl_lqr_current'] = Acl
            globals()['Q_lqr_current'] = Q
            globals()['R_lqr_current'] = R
            globals()['Nbar_lqr_current'] = nbar

        except Exception as e:
            print(f'LQR combined section error: {e}')


Q_text_lqr.observe(_draw_lqr_combined, names='value')
R_text_lqr.observe(_draw_lqr_combined, names='value')

display(widgets.VBox([
    widgets.HTML('<b>LQR weights</b> (row-major for Q: q11, q12, q21, q22)'),
    Q_text_lqr,
    widgets.HTML('R scalar: r'),
    R_text_lqr,
    out_lqr
]))

_draw_lqr_combined()